# PoliTok-DE: German election TikTok data

[Paper, arXiv v2](https://arxiv.org/html/2509.15860) · [Public ID release](https://huggingface.co/datasets/tomasruiz/PoliTok-DE). This notebook analyses released post IDs, availability snapshots, and Saxony annotation rows. Raw media, creator metadata, and comment rows are not in these public files. `DELETED` means not publicly accessible on a later check, not a confirmed policy violation.

## Local files and setup

**Purpose:** Confirm which public files are actually present on this computer. File presence does not imply that raw media or comment rows are included.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'downloaded_datasets').exists() and (ROOT.parent / 'downloaded_datasets').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'downloaded_datasets' / 'politok_de'
print('Local data folder:', DATA.resolve())
if DATA.exists():
    display(pd.DataFrame([{'file': p.name, 'size_MB': round(p.stat().st_size / 1e6, 2)}
                          for p in sorted(DATA.iterdir()) if p.is_file()]))
else:
    print('Local data folder is missing. Download the files first.')

Local data folder: /Users/wang/Documents/tiktok/downloaded_datasets/politok_de


,file,size_MB
0,federal_post_ids.parquet,9.70
1,saxony_annotations.parquet,0.03
2,saxony_post_ids.parquet,2.76


## 1. Structure and paper counts

**Purpose:** Compare the number of public post IDs with the revised paper, and distinguish annotation rows from annotated posts.

In [2]:
sax = pd.read_parquet(DATA / 'saxony_post_ids.parquet')
fed = pd.read_parquet(DATA / 'federal_post_ids.parquet')
ann = pd.read_parquet(DATA / 'saxony_annotations.parquet')
overview = pd.DataFrame([
    {'collection': 'Saxony 2024', 'public_rows': len(sax), 'unique_post_ids': sax.post_id.nunique(), 'paper_posts': 195373},
    {'collection': 'Federal 2025', 'public_rows': len(fed), 'unique_post_ids': fed.post_id.nunique(), 'paper_posts': 743588},
])
overview['difference_from_paper'] = overview.public_rows - overview.paper_posts
display(overview)
print('Overlapping post IDs:', len(set(sax.post_id) & set(fed.post_id)))
print('Annotation rows:', len(ann), 'unique annotated posts:', ann.post_id.nunique())
print('Public ID-file columns:', sax.columns.tolist(), fed.columns.tolist())

,collection,public_rows,unique_post_ids,paper_posts,difference_from_paper
0,Saxony 2024,195373,195373,195373,0
1,Federal 2025,743588,743588,743588,0


Overlapping post IDs: 0
Annotation rows: 935 unique annotated posts: 360
Public ID-file columns: ['post_id', 'collection', 'availability_2024_10_02', 'status_codes_2024_10_02', 'availability_2024_12_10', 'status_codes_2024_12_10', 'availability_2025_01_13', 'status_codes_2025_01_13'] ['post_id', 'collection', 'availability_2026_06_11', 'status_codes_2026_06_11']


**What this shows:** Both public post counts match the paper's v2 counts. The 935 annotation rows refer to only 360 posts; a row is not necessarily a distinct post.

## 2. Final availability checks

**Purpose:** Calculate accessibility at each collection's last reported recheck; show the denominator explicitly.

In [3]:
rows = []
for name, frame, col in [('Saxony 2024', sax, 'availability_2025_01_13'),
                         ('Federal 2025', fed, 'availability_2026_06_11')]:
    counts = frame[col].value_counts()
    rows.append({'collection': name, 'recheck_date': col.replace('availability_', ''),
                 'all_ids': len(frame), 'AVAILABLE': int(counts.get('AVAILABLE', 0)),
                 'DELETED': int(counts.get('DELETED', 0)),
                 'UNKNOWN_ERROR': int(counts.get('UNKNOWN_ERROR', 0)),
                 'deleted_share_of_all_ids_percent': 100 * counts.get('DELETED', 0) / len(frame)})
display(pd.DataFrame(rows).round(2))

,collection,recheck_date,all_ids,AVAILABLE,DELETED,UNKNOWN_ERROR,deleted_share_of_all_ids_percent
0,Saxony 2024,2025_01_13,195373,156742,36617,2014,18.74
1,Federal 2025,2026_06_11,743588,448487,295101,0,39.69


**What this shows:** The final `DELETED` shares are 18.74% (Saxony) and 39.69% (federal). The recheck dates differ, so the difference is not automatically an election effect. Saxony also has 2,014 `UNKNOWN_ERROR` IDs.

## 3. Saxony recheck waves

**Purpose:** Show how the number of conclusive checks changes across snapshots.

In [4]:
waves = []
for col in [x for x in sax.columns if x.startswith('availability_')]:
    counts = sax[col].value_counts()
    conclusive = int(counts.get('AVAILABLE', 0) + counts.get('DELETED', 0))
    waves.append({'recheck_date': col.replace('availability_', ''),
                  'AVAILABLE': int(counts.get('AVAILABLE', 0)), 'DELETED': int(counts.get('DELETED', 0)),
                  'NOT_RESCRAPED': int(counts.get('NOT_RESCRAPED', 0)),
                  'UNKNOWN_ERROR': int(counts.get('UNKNOWN_ERROR', 0)),
                  'conclusive_checks': conclusive,
                  'deleted_share_of_conclusive_percent': 100 * counts.get('DELETED', 0) / conclusive if conclusive else float('nan')})
display(pd.DataFrame(waves).round(2))

,recheck_date,AVAILABLE,DELETED,NOT_RESCRAPED,UNKNOWN_ERROR,conclusive_checks,deleted_share_of_conclusive_percent
0,2024_10_02,96474,6475,92420,4,102949,6.29
1,2024_12_10,88710,18568,87821,274,107278,17.31
2,2025_01_13,156742,36617,0,2014,193359,18.94


**What this shows:** The first wave has 92,420 `NOT_RESCRAPED` posts and only 102,949 conclusive results. Coverage differs across waves; a change in percentage is not a same-sample risk trend.

## 4. Human annotation and disagreement

**Purpose:** Separate individual votes from posts, and count posts with conflicting yes/no votes.

In [5]:
labels = ['is_political', 'is_saxony_election', 'is_intolerant',
          'is_hedonic_entertainment', 'is_eudaimonic_entertainment']
rows = []
for col in labels:
    votes = ann[col].str.lower().str.strip()
    valid = votes.isin(['yes', 'no'])
    per_post = ann.assign(vote=votes.where(valid)).groupby('post_id')['vote']
    rows.append({'label': col, 'valid_votes': int(valid.sum()),
                 'yes_share_of_valid_votes_percent': 100 * votes.eq('yes').sum() / valid.sum(),
                 'posts_with_disagreement': int((per_post.nunique(dropna=True) > 1).sum()),
                 'annotated_posts': ann.post_id.nunique()})
display(pd.DataFrame(rows).round(2))
display(ann.groupby('post_id').size().value_counts().sort_index()
        .rename_axis('annotation_rows_per_post').to_frame('post_count'))

,label,valid_votes,yes_share_of_valid_votes_percent,posts_with_disagreement,annotated_posts
0,is_political,932,85.62,20,360
1,is_saxony_election,932,19.10,33,360
2,is_intolerant,932,20.49,73,360
3,is_hedonic_entertainment,932,62.98,88,360
4,is_eudaimonic_entertainment,932,37.34,120,360


,post_count
annotation_rows_per_post,
1,60
2,119
3,90
4,88
5,3


**What this shows:** `is_intolerant=yes` accounts for about 20.49% of valid votes in this selected sample, not 20.49% of all election posts. Three posts have five annotation rows, while the paper says one to four annotators per post; the cause is not established.

## 5. Status-code combinations

**Purpose:** Inspect the technical codes accompanying inaccessible posts without treating all of them as platform takedowns.

In [6]:
for name, frame, av, sc in [('Saxony', sax, 'availability_2025_01_13', 'status_codes_2025_01_13'),
                            ('Federal', fed, 'availability_2026_06_11', 'status_codes_2026_06_11')]:
    codes = frame.loc[frame[av].eq('DELETED'), sc].fillna('')
    print(name, 'most common status-code combinations among DELETED IDs:')
    display(codes.value_counts().head(10).rename_axis('code_combination').to_frame('post_count'))

Saxony most common status-code combinations among DELETED IDs:


,post_count
code_combination,
status_deleted,9868
author_status|status_reviewing,7330
status_self_see,6720
author_status|cross_border_violation|status_audit_not_pass,3505
author_secret,1917
item_not_exist,1031
author_status|status_audit_not_pass,931
author_invalid|status_reviewing,680
author_status|status_deleted,565


Federal most common status-code combinations among DELETED IDs:


,post_count
code_combination,
status_deleted,80315
status_self_see,61698
author_status|status_reviewing,37071
author_invalid,17017
author_status|cross_border_violation|status_audit_not_pass,16906
author_secret,10684
author_status|status_audit_not_pass,7675
author_invalid|status_deleted,6623
author_secret|status_deleted,5775


**What this shows:** The paper uses code patterns to infer creator withdrawals versus platform removal. A simple `DELETED` flag alone does not identify the reason.

## Findings and limits

- The two ID files contain **195,373** and **743,588** distinct posts, matching paper v2; total **938,961**, with no overlap.
- The last reported `DELETED` snapshots contain **36,617** Saxony posts (18.74% of all IDs) and **295,101** federal posts (39.69%). Recheck timing differs.
- Saxony's first wave has **92,420** `NOT_RESCRAPED` posts; always specify the denominator.
- The annotation file has **935** rows for **360** posts. The intolerance-vote share is **20.49%** among valid votes in a selected subsample, not the full corpus.
- Public files omit creator metadata, captions, comments, and media; the paper's content-level analyses cannot be reproduced from these files alone.